In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Общая статистика

FILE = "students.csv"

# encoding="utf-8-sig" убирает BOM
# dtype=str — чтобы UUID и другие идентификаторы читались как текст
# keep_default_na=False — чтобы пустые поля остались пустыми строками, а не NaN
df = pd.read_csv(FILE, encoding="utf-8-sig", dtype=str, keep_default_na=False)

def print_value_counts_for_non_unique_columns(df):
    print(f"Всего записей: {len(df)}")
    print(f"Всего колонок: {len(df.columns)}")

""" 
    for col in df.columns:
        nunique = df[col].nunique(dropna=False)
        # Печатаем только те колонки, где есть повторяющиеся значения
        if nunique < len(df):
            print(f"\n=== {col} ===")
            print(f"Уникальных значений: {nunique}")
            print(df[col].value_counts(dropna=False).to_string())
"""
# Частоты по всем неуникальным колонкам
print_value_counts_for_non_unique_columns(df)

# Отдельно — самые важные для анализа поля
key_columns = [
    "EventType",
    "Status",
    "Result",
    "Role",
    "RequestType",
    "ActivityName",
    "TemplateVersion",
]

for col in key_columns:
    if col in df.columns:
        print(f"\n=== {col} ===")
        print(df[col].value_counts(dropna=False).to_string())

# Примеры точечных подсчётов
print("\n--- Точечные подсчёты ---")
print("Status = closed:", (df["Status"] == "closed").sum())
print("Role = system:", (df["Role"] == "system").sum())
print("EventType = task_completed:", (df["EventType"] == "task_completed").sum())
print("EventType = process_completed:", (df["EventType"] == "process_completed").sum())
print("Result = reject:", (df["Result"] == "reject").sum())
print("Result = return:", (df["Result"] == "return").sum())

Всего записей: 12162
Всего колонок: 15

=== EventType ===
EventType
task_created           4591
task_completed         4581
process_started        1000
automatic_completed    1000
process_completed       990

=== Status ===
Status
in_progress    4591
closed         4581
exec           1000
completed      1000
done            990

=== Result ===
Result
             6591
accept       4357
fulfilled     939
return        173
reject         51
rejected       51

=== Role ===
Role
               5581
requester      1181
specialist     1181
manager        1141
system         1000
finance         941
procurement     939
legal           182
auditor          16

=== RequestType ===
RequestType
standard    7575
contract    2629
urgent      1958

=== ActivityName ===
ActivityName
Проверка заявки               2362
Согласование руководителем    2306
Регистрация заявки            2000
Согласование бюджета          1882
Исполнение заявки             1880
Завершение                     990
Проверка д

In [ ]:
# Проверка по reject и rejected

# EventIndex нужен для сортировки событий внутри процесса
df["EventIndex"] = pd.to_numeric(df["EventIndex"], errors="coerce")

# Маски
task_reject = (df["EventType"] == "task_completed") & (df["Result"] == "reject")
proc_rejected = (df["EventType"] == "process_completed") & (df["Result"] == "rejected")

pids_task_reject = set(df.loc[task_reject, "ProcessInstanceId"])
pids_proc_rejected = set(df.loc[proc_rejected, "ProcessInstanceId"])

print("=== Сравнение ProcessInstanceId ===")
print(f"Процессов с task_completed/reject: {len(pids_task_reject)}")
print(f"Процессов с process_completed/rejected: {len(pids_proc_rejected)}")
print(f"Пересечение (есть и reject, и rejected): {len(pids_task_reject & pids_proc_rejected)}")
print(f"Только task reject, без process rejected: {len(pids_task_reject - pids_proc_rejected)}")
print(f"Только process rejected, без task reject: {len(pids_proc_rejected - pids_task_reject)}")

# Конкретные ID
only_task = pids_task_reject - pids_proc_rejected
only_proc = pids_proc_rejected - pids_task_reject

if only_task:
    print("\nProcessInstanceId только с task reject:")
    print("\n".join(sorted(only_task)))

if only_proc:
    print("\nProcessInstanceId только с process rejected:")
    print("\n".join(sorted(only_proc)))

# Функция подсчёта этапов для набора процессов
def stage_stats(df, pids):
    sub = df[df["ProcessInstanceId"].isin(pids)].copy()
    sub = sub.sort_values(["ProcessInstanceId", "EventIndex"])

    stats = sub.groupby("ProcessInstanceId").agg(
        events=("EventId", "count"),
        task_created=("EventType", lambda s: (s == "task_created").sum()),
        task_completed=("EventType", lambda s: (s == "task_completed").sum()),
        unique_activity_id=("ActivityId", "nunique"),
        unique_activity_name=("ActivityName", "nunique"),
        unique_task_id=("TaskId", "nunique"),
    )
    return stats

stats_reject = stage_stats(df, pids_task_reject)

print("\n=== Статистика по процессам с task_completed/reject ===")
print(stats_reject.to_string())

print("\n--- Распределения ---")

print("\nУникальные ActivityId:")
print(stats_reject["unique_activity_id"].value_counts().sort_index().to_string())

print("\nУникальные ActivityName:")
print(stats_reject["unique_activity_name"].value_counts().sort_index().to_string())

print("\nКоличество task_created:")
print(stats_reject["task_created"].value_counts().sort_index().to_string())

print("\nКоличество task_completed:")
print(stats_reject["task_completed"].value_counts().sort_index().to_string())

print("\nКоличество всех событий:")
print(stats_reject["events"].value_counts().sort_index().to_string())

# Проверка гипотезы: одинаковое ли число этапов у всех reject-процессов
print("\n=== Проверка гипотезы об одинаковом числе этапов ===")
print(
    "Все ли процессы с reject имеют одинаковое число уникальных ActivityId?",
    stats_reject["unique_activity_id"].nunique() == 1
)
print(
    "Все ли процессы с reject имеют одинаковое число уникальных ActivityName?",
    stats_reject["unique_activity_name"].nunique() == 1
)
print(
    "Все ли процессы с reject имеют одинаковое число task_created?",
    stats_reject["task_created"].nunique() == 1
)


=== Сравнение ProcessInstanceId ===
Процессов с task_completed/reject: 51
Процессов с process_completed/rejected: 51
Пересечение (есть и reject, и rejected): 51
Только task reject, без process rejected: 0
Только process rejected, без task reject: 0

=== Статистика по процессам с task_completed/reject ===
                                      events  task_created  task_completed  unique_activity_id  unique_activity_name  unique_task_id
ProcessInstanceId                                                                                                                   
0333cad4-e29a-584c-8f3d-3d70e7007b6c       7             2               2                   4                     4               3
07e43dc0-4c49-5801-ac8e-f561de037d8c       7             2               2                   4                     4               3
0f7b94b1-f6b7-5a03-ac33-49a7d4bf9152       7             2               2                   4                     4               3
1471b24f-b0b2-5a3f-9d70-d6448

In [4]:
df = pd.read_csv(FILE, encoding="utf-8-sig", dtype=str, keep_default_na=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True)
df["EventIndex"] = df["EventIndex"].astype(int)
df = df.sort_values(["ProcessInstanceId", "EventIndex"])

# --- Длительности заданий ---
tasks_created = (
    df[df["EventType"] == "task_created"]
    [["TaskId", "ProcessInstanceId", "ActivityName", "Timestamp"]]
    .rename(columns={"Timestamp": "created_ts"})
)
tasks_completed = (
    df[df["EventType"] == "task_completed"]
    [["TaskId", "Timestamp", "Result", "Role", "ActivityName"]]
    .rename(columns={"Timestamp": "completed_ts"})
)
tasks = tasks_created.merge(tasks_completed, on="TaskId", how="left")
tasks["duration"] = tasks["completed_ts"] - tasks["created_ts"]

def process_summary(group):
    start = group.loc[group["EventType"] == "process_started", "Timestamp"].min()
    end = group.loc[group["EventType"] == "process_completed", "Timestamp"].max()
    duration_cal = (end - start) if pd.notna(start) and pd.notna(end) else pd.NaT

    # Полный маршрут: каждое событие по порядку
    full_route = " -> ".join(group["ActivityName"].tolist())

    # Только человеческие этапы (task_created/task_completed), без автоматики и завершения
    human = group[group["EventType"].isin(["task_created", "task_completed"])]
    route_human = " -> ".join(
        f'{r.ActivityName}[{r.EventType.split("_")[1]}]'
        for r in human.itertuples()
    )

    # Счётчик прохождений каждого этапа (по task_created — одно создание = одно прохождение)
    created = group[group["EventType"] == "task_created"]["ActivityName"]
    pass_counts = created.value_counts().to_dict()

    return pd.Series({
        "RequestType": group["RequestType"].iloc[0],
        "Amount": group["Amount"].iloc[0],
        "start": start,
        "end": end,
        "duration_cal": duration_cal,
        "event_count": len(group),
        "full_route": full_route,
        "route_human": route_human,
        "n_check": pass_counts.get("Проверка заявки", 0),
        "n_manager": pass_counts.get("Согласование руководителем", 0),
        "n_finance": pass_counts.get("Согласование бюджета", 0),
        "n_legal": pass_counts.get("Проверка договора", 0),
        "n_procurement": pass_counts.get("Исполнение заявки", 0),
        "n_rework": pass_counts.get("Доработка заявки", 0),
        "n_audit": pass_counts.get("Внеплановая проверка", 0),
        "has_specialist": "Проверка заявки" in pass_counts,
        "has_manager": "Согласование руководителем" in pass_counts,
        "has_finance": "Согласование бюджета" in pass_counts,
        "has_legal": "Проверка договора" in pass_counts,
        "has_procurement": "Исполнение заявки" in pass_counts,
        "has_audit": "Внеплановая проверка" in pass_counts,
        "returns": (group["Result"] == "return").sum(),
        "rejects": (group["Result"] == "reject").sum(),
        "completed": (group["EventType"] == "process_completed").any(),
        "fulfilled": (group["Result"] == "fulfilled").sum(),
        "rejected": (group["Result"] == "rejected").sum(),
    })

proc = df.groupby("ProcessInstanceId").apply(process_summary).reset_index()
proc.to_csv("process_summary.csv", index=False, encoding="utf-8-sig")

# Посмотрим на первые строки и общую статистику
print(proc.head())
print("\n--- Описательные статистики ---")
print(proc[["duration_cal", "event_count", "returns", "rejects"]].describe())

print(tasks.head())
# Сохраним для дальнейшего анализа
proc.to_csv("process_summary.csv", index=False, encoding="utf-8-sig")
tasks.to_csv("tasks_with_durations.csv", index=False, encoding="utf-8-sig")


                      ProcessInstanceId RequestType    Amount  \
0  007212e1-7659-559a-8f7d-4871296b8fd5    standard   9861.11   
1  00721b96-70ca-5e08-a103-94a4138d1e92    standard  23571.66   
2  0144d0f8-91fd-5341-88b8-08fe48b62820    standard  87269.45   
3  016ff013-1c3b-52d9-9ab8-2cdb4085ea24    standard  44229.49   
4  01e37708-5e54-5866-9a6c-77e8e2f1936e    standard   5724.72   

                      start                       end    duration_cal  \
0 2026-06-15 06:00:00+00:00 2026-06-16 13:48:47+00:00 1 days 07:48:47   
1 2026-06-29 06:00:00+00:00 2026-06-30 06:26:56+00:00 1 days 00:26:56   
2 2026-06-17 06:00:00+00:00 2026-06-18 11:26:56+00:00 1 days 05:26:56   
3 2026-02-16 06:00:00+00:00 2026-02-17 09:33:05+00:00 1 days 03:33:05   
4 2026-05-06 10:13:49+00:00 2026-05-07 11:23:15+00:00 1 days 01:09:26   

   event_count                                         full_route  \
0           17  Регистрация заявки -> Регистрация заявки -> Пр...   
1           11  Регистрация заяв

Анализ систематизированных данных

In [5]:
proc = pd.read_csv("process_summary.csv", encoding="utf-8-sig", keep_default_na=False)
tasks = pd.read_csv("tasks_with_durations.csv", encoding="utf-8-sig", keep_default_na=False)

def canon(full_route):
    parts = [p.strip() for p in full_route.split("->")]
    out = []
    for p in parts:
        if not out or out[-1] != p:
            out.append(p)
    return " -> ".join(out)

proc["route_canon"] = proc["full_route"].apply(canon)

route_counts = (
    proc.groupby(["RequestType", "route_canon"])
        .agg(n=("ProcessInstanceId", "count"),
             n_with_rework=("n_rework", lambda s: (pd.to_numeric(s) > 0).sum()),
             n_with_audit=("n_audit", lambda s: (pd.to_numeric(s) > 0).sum()),
             n_rejected=("rejected", "sum"))
        .sort_values(["RequestType", "n"], ascending=[True, False])
)
print(route_counts.to_string())


                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      n  n_with_rework  n_with_audit  n_rejected
RequestType route_canon                                                                                                                                                                                                                                                

In [6]:
from datetime import datetime, time, timedelta
import pytz

df = pd.read_csv(FILE, encoding="utf-8-sig", dtype=str, keep_default_na=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True)

WORK_TZ = pytz.timezone("Europe/Moscow")   # UTC+3
WS, WE = time(9, 0), time(18, 0)

def working_hours(s, e):
    if pd.isna(s) or pd.isna(e): return np.nan
    s, e = s.astimezone(WORK_TZ), e.astimezone(WORK_TZ)
    if e <= s: return 0.0
    total, d, end_d = 0.0, s.date(), e.date()
    while d <= end_d:
        if d.weekday() < 5:                          # Пн–Пт
            ds = WORK_TZ.localize(datetime.combine(d, WS))
            de = WORK_TZ.localize(datetime.combine(d, WE))
            a, b = max(s, ds), min(e, de)
            if b > a: total += (b - a).total_seconds()
        d += timedelta(days=1)
    return total / 3600.0

# --- Склейка task_created + task_completed по TaskId ---
created = (df[df.EventType == "task_created"]
           [["TaskId","ProcessInstanceId","ActivityName","Timestamp"]]
           .rename(columns={"Timestamp":"created_ts","ActivityName":"activity"}))
done    = (df[df.EventType == "task_completed"]
           [["TaskId","Timestamp","Result","Role","UserId","ActivityName"]]
           .rename(columns={"Timestamp":"completed_ts"}))
tasks = created.merge(done.drop(columns="ActivityName"), on="TaskId", how="left")

tasks["cal_h"]  = (tasks["completed_ts"] - tasks["created_ts"]).dt.total_seconds()/3600
tasks["work_h"] = [working_hours(c, k) for c, k in zip(tasks["created_ts"], tasks["completed_ts"])]

# ========== 1. РОЛИ ==========
print("="*78)
print("РАСПРЕДЕЛЕНИЕ РОЛЕЙ ПО ЭТАПАМ (только закрытые задачи)")
print("="*78)
closed = tasks.dropna(subset=["completed_ts"])
role_counts = closed.groupby(["activity","Role"]).size().unstack(fill_value=0)
role_pct    = role_counts.div(role_counts.sum(axis=1), axis=0) * 100

print("\nАбсолютные количества (строки = этап, столбцы = роль):")
print(role_counts.to_string())
print("\nПроценты (%):")
print(role_pct.round(1).to_string())

# ========== 2. ДЛИТЕЛЬНОСТЬ ==========
print("\n" + "="*78)
print("ДЛИТЕЛЬНОСТЬ ЗАКРЫТЫХ ЗАДАЧ ПО ЭТАПАМ (часы)")
print("="*78)
stats = closed.groupby("activity").agg(
    n          = ("TaskId","count"),
    cal_min    = ("cal_h","min"),
    cal_p25    = ("cal_h", lambda x: x.quantile(0.25)),
    cal_med    = ("cal_h","median"),
    cal_p75    = ("cal_h", lambda x: x.quantile(0.75)),
    cal_p95    = ("cal_h", lambda x: x.quantile(0.95)),
    cal_max    = ("cal_h","max"),
    work_med   = ("work_h","median"),
    work_p95   = ("work_h", lambda x: x.quantile(0.95)),
    work_max   = ("work_h","max"),
).round(2)
print(stats.to_string())

# ========== 3. ОЖИДАЕМАЯ РОЛЬ vs ФАКТ ==========
print("\n" + "="*78)
print("НЕСООТВЕТСТВИЕ РОЛИ ЭТАПУ")
print("="*78)
expected = {
    "Проверка заявки":              "specialist",
    "Согласование руководителем":   "manager",
    "Доработка заявки":             "requester",
    "Согласование бюджета":         "finance",
    "Проверка договора":            "legal",
    "Исполнение заявки":            "procurement",
    "Внеплановая проверка":         "auditor",
}
closed = closed.copy()
closed["expected_role"] = closed["activity"].map(expected)
wrong = closed[closed["expected_role"].notna() & (closed["Role"] != closed["expected_role"])]
print(f"Всего закрытий с явно не той ролью: {len(wrong)}")
if len(wrong):
    print(wrong.groupby(["activity","expected_role","Role"]).size()
          .rename("count").to_string())

# ========== 4. ОТКРЫТЫЕ ЗАДАЧИ (нижний предел) ==========
print("\n" + "="*78)
print("ОТКРЫТЫЕ ЗАДАЧИ НА СРЕЗ 2026-07-01 00:00 UTC+3 (нижний предел)")
print("="*78)
now = pd.Timestamp("2026-07-01 00:00:00", tz="Europe/Moscow").tz_convert("UTC")
op = tasks[tasks["completed_ts"].isna()].copy()
op["elapsed_cal"]  = (now - op["created_ts"]).dt.total_seconds()/3600
op["elapsed_work"] = [working_hours(c, now) for c in op["created_ts"]]
print(op[["ProcessInstanceId","activity","created_ts","elapsed_cal","elapsed_work"]]
      .sort_values("elapsed_work", ascending=False).round(1).to_string())

РАСПРЕДЕЛЕНИЕ РОЛЕЙ ПО ЭТАПАМ (только закрытые задачи)

Абсолютные количества (строки = этап, столбцы = роль):
Role                        auditor  finance  legal  manager  procurement  requester  specialist
activity                                                                                        
Внеплановая проверка             16        0      0        0            0          0           0
Доработка заявки                  0        0      0        0            0        173           0
Исполнение заявки                 0        0      0        0          939          0           0
Проверка договора                 0        0    182        0            0          0           0
Проверка заявки                   0        0      0        0            0          0        1181
Согласование бюджета              0      941      0        0            0          0           0
Согласование руководителем        0        0      0     1141            0          8           0

Проценты (%):
R

C:\Users\Akantr\AppData\Local\Temp\ipykernel_16608\1037033672.py:97: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .sort_values("elapsed_work", ascending=False).round(1).to_string())


In [7]:
import pandas as pd, numpy as np, pytz
from datetime import datetime, time, timedelta
FILE = "students.csv"

df = pd.read_csv(FILE, encoding="utf-8-sig", dtype=str, keep_default_na=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True)
df["EventIndex"] = df["EventIndex"].astype(int)

WORK_TZ = pytz.timezone("Europe/Moscow"); WS, WE = time(9,0), time(18,0)
def wh(s, e):
    if pd.isna(s) or pd.isna(e): return np.nan
    s, e = s.astimezone(WORK_TZ), e.astimezone(WORK_TZ)
    if e <= s: return 0.0
    total, d, end = 0.0, s.date(), e.date()
    while d <= end:
        if d.weekday() < 5:
            ds = WORK_TZ.localize(datetime.combine(d, WS)); de = WORK_TZ.localize(datetime.combine(d, WE))
            a, b = max(s, ds), min(e, de)
            if b > a: total += (b-a).total_seconds()
        d += timedelta(days=1)
    return total/3600

created = (df[df.EventType=="task_created"][["TaskId","ProcessInstanceId","ActivityName","Timestamp"]]
           .rename(columns={"Timestamp":"created_ts","ActivityName":"activity"}))
done    = (df[df.EventType=="task_completed"][["TaskId","Timestamp","Result","Role","UserId"]]
           .rename(columns={"Timestamp":"completed_ts"}))
tasks = created.merge(done, on="TaskId", how="left")
tasks["work_h"] = [wh(c,k) for c,k in zip(tasks.created_ts, tasks.completed_ts)]
tasks["mmss_same"] = (tasks.created_ts.dt.minute   == tasks.completed_ts.dt.minute) & \
                     (tasks.created_ts.dt.second   == tasks.completed_ts.dt.second)
tasks["offset_h"]  = (tasks.completed_ts - tasks.created_ts).dt.total_seconds()/3600

rows = []

# --- A1 / A2: self-approval и role-substitution ---
initiator = df[df.EventType=="process_started"][["ProcessInstanceId","UserId"]].rename(columns={"UserId":"initiator"})
bad = (df[(df.EventType=="task_completed") &
          (df.ActivityName=="Согласование руководителем") &
          (df.Role=="requester")].merge(initiator, on="ProcessInstanceId"))
for _, r in bad.iterrows():
    typ = "self_approval" if r.UserId == r.initiator else "role_substitution"
    exp = (f"Задача «Согласование руководителем» (TaskId={r.TaskId}, Result={r.Result}) "
           f"закрыта пользователем {r.UserId} с ролью requester; "
           f"{'это инициатор процесса' if typ=='self_approval' else 'это другой requester, не руководитель'}.")
    rows.append({"ProcessInstanceId": r.ProcessInstanceId, "AnomalyType": typ, "Explanation": exp})

# --- B: scripted_duration (целые часы + сохранённые mm:ss) ---
closed_tasks = tasks.dropna(subset=["completed_ts", "created_ts"]).copy()

closed_tasks["is_whole_hours"] = (
    closed_tasks["offset_h"].apply(lambda x: abs(x - round(x)) < 1e-6)
)

scr = closed_tasks[
    closed_tasks["mmss_same"]
    & closed_tasks["is_whole_hours"]
    & (closed_tasks["offset_h"] >= 1)
]
for _, r in scr.iterrows():
    rows.append({"ProcessInstanceId": r.ProcessInstanceId, "AnomalyType": "scripted_duration",
                 "Explanation": (f"Этап «{r.activity}» (TaskId={r.TaskId}) закрыт через ровно "
                                 f"{int(r.offset_h)} ч ({r.work_h:.3f} раб.ч).")})

# --- C: instant_approval_chain (>=3 этапов < 0.5 мин в одном процессе) ---
tasks["sub30s"] = tasks.work_h < 0.5/60
fast = (tasks[tasks.sub30s].groupby("ProcessInstanceId")
        .agg(n_fast=("TaskId","count"), stages=("activity", lambda s: sorted(set(s)))))
fast = fast[fast.n_fast >= 3]
for pid, r in fast.iterrows():
    rows.append({"ProcessInstanceId": pid, "AnomalyType": "instant_approval_chain",
                 "Explanation": (f"{r.n_fast} задач в {len(r.stages)} этапах ({', '.join(r.stages)}) "
                                 f"закрыты менее чем за 30 с каждая.")})

# --- структурные нарушения из прошлого шага (без руководителя / исполнение раньше бюджета) ---
act = df.groupby(["ProcessInstanceId","EventIndex"]).first().reset_index()  # не нужно, оставим при желании
seq = (df.sort_values(["ProcessInstanceId","EventIndex"])
       .groupby("ProcessInstanceId")["ActivityName"]
       .apply(lambda s: list(s)).reset_index(name="seq"))
mandatory = {"standard": ["Проверка заявки","Согласование руководителем","Согласование бюджета","Исполнение заявки","Завершение"],
             "urgent":   ["Проверка заявки","Согласование руководителем","Согласование бюджета","Исполнение заявки","Завершение"],
             "contract": ["Проверка заявки","Согласование руководителем","Согласование бюджета","Проверка договора","Исполнение заявки","Завершение"]}
init_type = df.groupby("ProcessInstanceId")["RequestType"].first()
for pid, s in zip(seq.ProcessInstanceId, seq.seq):
    t = init_type[pid]
    stages = set(s)
    if "Согласование руководителем" not in stages and "Завершение" in stages:
        rows.append({"ProcessInstanceId": pid, "AnomalyType": "missing_manager",
                     "Explanation": "Процесс дошёл до «Завершение» без этапа «Согласование руководителем»."})
    if "Исполнение заявки" in stages and "Согласование бюджета" in stages:
        if s.index("Исполнение заявки") < s.index("Согласование бюджета"):
            rows.append({"ProcessInstanceId": pid, "AnomalyType": "execution_before_budget",
                         "Explanation": "«Исполнение заявки» выполнено раньше «Согласование бюджета»."})
    if t == "contract" and "Исполнение заявки" in stages and "Проверка договора" in stages:
        if s.index("Исполнение заявки") < s.index("Проверка договора"):
            rows.append({"ProcessInstanceId": pid, "AnomalyType": "execution_before_legal",
                         "Explanation": "Для contract-заявки «Исполнение» выполнено раньше «Проверки договора»."})

reworks = (df[df.ActivityName == "Доработка заявки"]
           .groupby("ProcessInstanceId").size())
over = reworks[reworks >= 4]
for pid, n in over.items():
    rows.append({"ProcessInstanceId": pid, "AnomalyType": "excessive_rework",
                 "Explanation": (f"Этап «Доработка заявки» выполнялся {n} раз.")})

# Ищем task_created того же этапа, что и предыдущее событие в процессе,
# при условии, что предыдущее событие не было возвратом (Result != "return").
ev = df.sort_values(["ProcessInstanceId", "EventIndex"]).copy()
prev_result = ev.groupby("ProcessInstanceId")["Result"].shift()
prev_act    = ev.groupby("ProcessInstanceId")["ActivityName"].shift()

dup_mask = (
    (ev.EventType == "task_created")
    & (ev.ActivityName == prev_act)
    & (prev_result != "return")
)
dup_no_return = ev[dup_mask]

# Одна запись на процесс — с перечислением всех повторных созданий
for pid, g in dup_no_return.groupby("ProcessInstanceId"):
    details = "; ".join(
        f"«{r.ActivityName}» (EventIndex={r.EventIndex})"
        for r in g.itertuples()
    )
    rows.append({
        "ProcessInstanceId": pid,
        "AnomalyType": "duplicate_activity_without_return",
        "Explanation": (
            f"Этап создан повторно без возврата на доработку (Result = return). "
        ),
    })

out = pd.DataFrame(rows).drop_duplicates()
out.to_csv("anomalies.csv", index=False, encoding="utf-8", quoting=1)
print(f"Строк в anomalies.csv: {len(out)}; уникальных процессов: {out.ProcessInstanceId.nunique()}")
print(out.groupby("AnomalyType").size().sort_values(ascending=False).to_string())

Строк в anomalies.csv: 77; уникальных процессов: 66
AnomalyType
scripted_duration                    19
missing_manager                      16
duplicate_activity_without_return     8
excessive_rework                      8
execution_before_budget               8
instant_approval_chain                7
role_substitution                     6
execution_before_legal                3
self_approval                         2


In [8]:
df = pd.read_csv(FILE, encoding="utf-8-sig", dtype=str, keep_default_na=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True)

# --- построение tasks (как в предыдущем ответе) ---
created = (df[df.EventType=="task_created"][["TaskId","ProcessInstanceId","ActivityName","Timestamp"]]
           .rename(columns={"Timestamp":"created_ts","ActivityName":"activity"}))
done = (df[df.EventType=="task_completed"][["TaskId","Timestamp","Result","Role","UserId"]]
        .rename(columns={"Timestamp":"completed_ts"}))
tasks = created.merge(done, on="TaskId", how="left")
tasks["cal_h"] = (tasks.completed_ts - tasks.created_ts).dt.total_seconds()/3600

WORK_TZ = pytz.timezone("Europe/Moscow"); WS, WE = time(9,0), time(18,0)
def wh(s,e):
    if pd.isna(s) or pd.isna(e): return np.nan
    s,e = s.astimezone(WORK_TZ), e.astimezone(WORK_TZ)
    if e<=s: return 0.0
    total,d,end = 0.0, s.date(), e.date()
    while d<=end:
        if d.weekday()<5:
            ds=WORK_TZ.localize(datetime.combine(d,WS)); de=WORK_TZ.localize(datetime.combine(d,WE))
            a,b=max(s,ds),min(e,de)
            if b>a: total+=(b-a).total_seconds()
        d+=timedelta(days=1)
    return total/3600
tasks["work_h"] = [wh(c,k) for c,k in zip(tasks.created_ts, tasks.completed_ts)]

# ============================================================
# 1. SELF-APPROVAL: requester закрыл согласование руководителя
# ============================================================
bad = df[(df.EventType=="task_completed") &
         (df.ActivityName=="Согласование руководителем") &
         (df.Role=="requester")]

# кто инициировал процесс
initiator = (df[df.EventType=="process_started"]
             [["ProcessInstanceId","UserId"]].rename(columns={"UserId":"initiator"}))
bad = bad.merge(initiator, on="ProcessInstanceId", how="left")
bad["is_initiator"] = bad.UserId == bad.initiator

print("Самосогласование (requester закрыл задачу руководителя):")
print(bad[["ProcessInstanceId","TaskId","Timestamp","UserId","initiator","is_initiator","Result"]]
      .to_string(index=False))
print("\nИз них совпадает с инициатором процесса:",
      bad.is_initiator.sum(), "из", len(bad))

# что было дальше в этих процессах
for pid in bad.ProcessInstanceId.unique():
    print(f"\n--- Процесс {pid} ---")
    print(df[df.ProcessInstanceId==pid]
          .sort_values("EventIndex")
          [["EventIndex","EventType","ActivityName","Timestamp","UserId","Role","Result"]]
          .to_string(index=False))

# ============================================================
# 2. ДЛИТЕЛЬНОСТНЫЕ ВЫБРОСЫ внутри ActivityName
# ============================================================
def flag(g):
    q1,q3 = g.work_h.quantile([0.25,0.75]); iqr=q3-q1
    upper = q3 + 3*iqr
    med = g.work_h.median()
    m = (g.work_h > upper) & (g.work_h > 5*med)     # высокий и относительно медианы
    m |= g.work_h > 10*med                           # экстремальный
    m |= g.work_h < 1/60                             # < 1 минуты
    return m

tasks["flag"] = False
for act, g in tasks.dropna(subset=["work_h"]).groupby("activity"):
    tasks.loc[g.index, "flag"] = flag(g)

out = tasks[tasks.flag].sort_values(["activity","work_h"])
print(f"\nВыбросов по длительности: {len(out)}")
print(out[["ProcessInstanceId","activity","created_ts","completed_ts",
           "cal_h","work_h","Role","UserId","Result"]].round(3).to_string(index=False))

# ============================================================
# 3. SEPARATION OF DUTIES по UserId + нагрузка по ролям
# ============================================================
done_df = df[df.EventType=="task_completed"]

# один UserId закрыл задачи разных ролей в одном процессе
multi = (done_df.groupby(["ProcessInstanceId","UserId"])
         .agg(roles=("Role", lambda s: sorted(set(s))), n=("TaskId","count"))
         .reset_index())
multi = multi[multi.roles.apply(len) > 1]
print(f"\nПроцессов, где один UserId закрывал разные роли: {multi.ProcessInstanceId.nunique()}")
print(multi.to_string(index=False))

# сколько разных пользователей реально работает в каждой роли
print("\nЧисло различных UserId по ролям:")
print(done_df.groupby("Role")["UserId"].nunique().sort_values().to_string())

# десятка самых активных в роли specialist (проверить «одного на всё»)
print("\nТоп-10 specialist по числу закрытий:")
print(done_df[done_df.Role=="specialist"].UserId.value_counts().head(10).to_string())

Самосогласование (requester закрыл задачу руководителя):
                   ProcessInstanceId                               TaskId                 Timestamp                               UserId                            initiator  is_initiator Result
65ba6ad5-5465-5047-b7e6-cd9f79f3b7dc 7efceedf-bd91-53c7-8c1c-fd39bdbe5a34 2026-01-29 08:04:49+00:00 d84c351b-da4b-5c13-a5d8-49746761a078 5da4329a-e781-5f3d-bd84-038cf2f1634d         False reject
59ca16a9-80c3-5547-ac6e-babd341ef2b6 dbb4577f-a91b-572e-94fc-f9adbfcd34ea 2026-03-16 14:06:06+00:00 5da4329a-e781-5f3d-bd84-038cf2f1634d 5da4329a-e781-5f3d-bd84-038cf2f1634d          True accept
583c040c-badc-5b6c-b9af-14cbde34229c 0be68b33-bb04-55f4-a129-1e0a2aae8260 2026-03-23 10:11:09+00:00 e6593541-c219-5cfa-8cdb-03d310ae9381 5da4329a-e781-5f3d-bd84-038cf2f1634d         False accept
35097b8a-4b7a-5efa-9645-b3d552458029 79ad2485-9a31-56f1-8303-b6b0d10d00bb 2026-04-21 14:36:30+00:00 0cebb952-3bf8-568b-bde6-ea14e4d89c8b 5da4329a-e781-5f3d-bd84-03

C:\Users\Akantr\AppData\Local\Temp\ipykernel_16608\1617978125.py:73: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  "cal_h","work_h","Role","UserId","Result"]].round(3).to_string(index=False))


In [ ]:
# Повтор этапов без return
ev = df.sort_values(["ProcessInstanceId","EventIndex"]).copy()
prev_result = ev.groupby("ProcessInstanceId")["Result"].shift()
prev_act    = ev.groupby("ProcessInstanceId")["ActivityName"].shift()

dup_no_return = ev[
    (ev.EventType == "task_created")
    & (ev.ActivityName == prev_act)          # тот же этап, что и раньше
    & (prev_result != "return")              # без возврата между ними
]
print(dup_no_return[["ProcessInstanceId","EventIndex","ActivityName"]])


                         ProcessInstanceId EventIndex     ActivityName
4079  17df6b95-3ad5-5a45-8076-161f76d69138          5  Проверка заявки
1250  2fc494c8-5171-54d6-bfa1-ff40af1a2b24          5  Проверка заявки
4127  47cc5aa0-3538-5136-8163-8000c38df6d6          5  Проверка заявки
3617  5a4d84fa-1b52-522d-aaea-ee726c3b6ab5          5  Проверка заявки
1038  7f0d02c6-bac1-5d0b-9763-c1edf66d9c30          5  Проверка заявки
7311  864a927e-6c22-54ee-b2ea-a55a9c591f7b          5  Проверка заявки
5222  caa40070-3d4a-5576-9b45-ee9be7a679a0          5  Проверка заявки
8435  fc77f43a-645d-5045-8a0f-cd9719de69d3          5  Проверка заявки


In [ ]:
# Финансовая статистика
am = df.groupby("ProcessInstanceId").agg(
    RequestType=("RequestType","first"), Amount=("Amount","first"))
am["Amount"] = am.Amount.astype(float)
def iqr_flag(g):
    q1,q3 = g.Amount.quantile([.25,.75]); iqr=q3-q1
    return g[(g.Amount < q1-3*iqr) | (g.Amount > q3+3*iqr)]
out = am.groupby("RequestType").apply(iqr_flag)
print(out)



                                                     Amount
RequestType ProcessInstanceId                              
contract    8a347705-d655-5dc0-a51a-fe496ac782d6  144116.90
            a7cb0d9c-b0da-5cb0-8f6b-b4ab3d3669b6  169084.17
            ca470447-19aa-54e4-9159-80ca2ac7faf7  200492.29
            debf8692-3d0b-5462-88c7-7f020e3790ad  172445.10
            e137caef-5528-5e65-9f0e-69b44b102c8e  175139.73
            fe556890-4fab-5018-a64f-b452b91e4c60  143099.75
standard    17e28d48-3533-5381-a5d1-e86c65310f13  164243.73
            3284416a-18f7-5e21-a5e9-228ed57c3fc9  204909.59
            47cc5aa0-3538-5136-8163-8000c38df6d6  239246.01
            48c23829-3f53-59b3-83ee-50f9aead7d15  212014.51
            5cdc1eb7-0966-5358-b915-71247c4dedf5  151442.84
            bcf71d17-0a83-53fb-b1f1-3f6a4d2dfb5e  151150.98
            d0c0215f-7d87-55f8-8b2e-69fc115fe54b  143621.12
            e8091d97-a559-5636-aaf2-333444a22772  255785.71
urgent      94889274-ee97-5d0b-b06f-c329